[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rsasaki0109/slam-handbook-python/blob/main/part2_practice/ch07_visual_slam/01_camera_and_epipolar.ipynb)

# Chapter 7: Visual SLAM — カメラモデル、特徴点、エピポーラ幾何、バンドル調整

**SLAM Handbook Chapter 7** — Visual SLAMの基本要素をPythonで実装。

## このNotebookで学ぶこと

1. **Pinhole Camera Model** — 3D→2D射影、カメラ内部パラメータ
2. **Reprojection Error** — 再投影誤差の定義
3. **特徴点検出とマッチング** — Harris/ORBの概念
4. **エピポーラ幾何** — Essential Matrix、8点法
5. **バンドル調整 (BA)** — カメラポーズと3D点の同時最適化

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import least_squares

np.set_printoptions(precision=4, suppress=True)
%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

# 3Dプロット用（バージョン互換）
try:
    from mpl_toolkits.mplot3d import Axes3D
except ImportError:
    pass

## 7.1 Pinhole Camera Model

**SLAM Handbook Section 7.3.1**: 3D点 $\mathbf{x}^c = [x, y, z]^\top$ をカメラ座標系で2Dピクセル $\mathbf{z} = [u, v]^\top$ に射影:

$$\pi_p(\mathbf{x}^c, \boldsymbol{\xi}) = \begin{bmatrix} f_u \frac{x}{z} + u_0 \\ f_v \frac{y}{z} + v_0 \end{bmatrix}$$

カメラ内部パラメータ: $\boldsymbol{\xi}_p = [f_u, f_v, u_0, v_0]^\top$

逆射影（depth $d$ が必要）: $\pi^{-1}(\mathbf{z}, \boldsymbol{\xi}) \sim \begin{bmatrix} (u-u_0)/f_u \\ (v-v_0)/f_v \\ 1 \end{bmatrix}$

In [ ]:
# =============================================================
# Pinhole Camera Model の実装
# =============================================================

class PinholeCamera:
    def __init__(self, fu, fv, u0, v0):
        self.fu, self.fv = fu, fv
        self.u0, self.v0 = u0, v0
        self.K = np.array([[fu, 0, u0], [0, fv, v0], [0, 0, 1]])
    
    def project(self, p_cam):
        """3D点(カメラ座標) → 2Dピクセル  p_cam: (N,3) or (3,)"""
        p = np.atleast_2d(p_cam)
        u = self.fu * p[:, 0] / p[:, 2] + self.u0
        v = self.fv * p[:, 1] / p[:, 2] + self.v0
        return np.column_stack([u, v])
    
    def unproject(self, z, depth=1.0):
        """2Dピクセル → 3D方向ベクトル (depth=1で正規化)"""
        z = np.atleast_2d(z)
        x = (z[:, 0] - self.u0) / self.fu
        y = (z[:, 1] - self.v0) / self.fv
        return depth * np.column_stack([x, y, np.ones(len(z))])

# SO(3) ユーティリティ
def so3_exp(v):
    phi = np.linalg.norm(v)
    if phi < 1e-10: return np.eye(3) + np.array([[0,-v[2],v[1]],[v[2],0,-v[0]],[-v[1],v[0],0]])
    a = v/phi; aw = np.array([[0,-a[2],a[1]],[a[2],0,-a[0]],[-a[1],a[0],0]])
    return np.eye(3) + np.sin(phi)*aw + (1-np.cos(phi))*(aw@aw)

# テスト: 3D点を射影
cam = PinholeCamera(fu=500, fv=500, u0=320, v0=240)

# ワールド座標の3D点
points_3d = np.array([
    [0, 0, 5], [1, 0, 5], [-1, 0, 5],
    [0, 1, 5], [0, -1, 5], [0.5, 0.5, 3],
    [-0.5, -0.5, 7], [1.5, 1, 4], [-1, 1, 6],
])

# カメラ1: 原点
R1 = np.eye(3)
t1 = np.zeros(3)
p_cam1 = (R1 @ points_3d.T).T + t1  # ワールド→カメラ
z1 = cam.project(p_cam1)

# カメラ2: 少し右に移動 + 回転
R2 = so3_exp([0, 0.1, 0])
t2 = np.array([0.5, 0, 0])
p_cam2 = (R2 @ points_3d.T).T + t2
z2 = cam.project(p_cam2)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, z, title in [(axes[0], z1, 'Camera 1 (原点)'),
                       (axes[1], z2, 'Camera 2 (右に移動+回転)')]:
    ax.scatter(z[:, 0], z[:, 1], c='blue', s=50)
    for i, (u, v) in enumerate(z):
        ax.annotate(f'P{i}', (u, v), textcoords='offset points', xytext=(5, 5), fontsize=8)
    ax.set_xlim(0, 640); ax.set_ylim(480, 0)  # 画像座標系
    ax.set_xlabel('u [px]'); ax.set_ylabel('v [px]')
    ax.set_title(title, fontweight='bold')
    ax.set_aspect('equal'); ax.grid(True, alpha=0.3)

plt.suptitle('Pinhole Camera: 3D点の2D射影', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

## 7.2 エピポーラ幾何と Essential Matrix

2つのカメラ間の幾何的関係。対応点 $\mathbf{z}_1, \mathbf{z}_2$ が満たす **エピポーラ制約**:

$$\hat{\mathbf{z}}_2^\top \mathbf{E} \, \hat{\mathbf{z}}_1 = 0$$

ここで $\hat{\mathbf{z}} = \mathbf{K}^{-1}\mathbf{z}$ は正規化座標、$\mathbf{E} = [\mathbf{t}]_\times \mathbf{R}$ がEssential Matrix。

### 8点法 (Eight-Point Algorithm)
8組以上の対応点から $\mathbf{E}$ を推定し、$\mathbf{R}, \mathbf{t}$ を復元。

In [ ]:
# =============================================================
# Essential Matrix の推定 (8点法)
# =============================================================

def skew(v):
    return np.array([[0, -v[2], v[1]], [v[2], 0, -v[0]], [-v[1], v[0], 0]])

# 正規化座標に変換
K_inv = np.linalg.inv(cam.K)
z1_h = np.column_stack([z1, np.ones(len(z1))])  # homogeneous
z2_h = np.column_stack([z2, np.ones(len(z2))])
z1_norm = (K_inv @ z1_h.T).T  # 正規化座標
z2_norm = (K_inv @ z2_h.T).T

# 真のEssential Matrix
E_true = skew(t2) @ R2
print("真のEssential Matrix:")
print(E_true)

# エピポーラ制約の検証: z2^T E z1 = 0
for i in range(len(points_3d)):
    val = z2_norm[i] @ E_true @ z1_norm[i]
    # print(f"  点{i}: z2^T E z1 = {val:.6f}")

# 8点法の実装
def eight_point_algorithm(pts1, pts2):
    """正規化座標の対応点からEssential Matrixを推定"""
    n = len(pts1)
    A = np.zeros((n, 9))
    for i in range(n):
        x1, y1, _ = pts1[i]
        x2, y2, _ = pts2[i]
        A[i] = [x2*x1, x2*y1, x2, y2*x1, y2*y1, y2, x1, y1, 1]
    
    _, _, Vt = np.linalg.svd(A)
    E = Vt[-1].reshape(3, 3)
    
    # E のrank-2制約を強制 (特異値を [s1, s2, 0] にする)
    U, S, Vt = np.linalg.svd(E)
    S = np.array([(S[0]+S[1])/2, (S[0]+S[1])/2, 0])
    E = U @ np.diag(S) @ Vt
    return E

# ノイズ付きで推定
np.random.seed(42)
sigma_px = 1.0  # 1ピクセルのノイズ
z1_noisy = z1 + np.random.normal(0, sigma_px, z1.shape)
z2_noisy = z2 + np.random.normal(0, sigma_px, z2.shape)
z1n_noisy = (K_inv @ np.column_stack([z1_noisy, np.ones(len(z1))]).T).T
z2n_noisy = (K_inv @ np.column_stack([z2_noisy, np.ones(len(z2))]).T).T

E_est = eight_point_algorithm(z1n_noisy, z2n_noisy)

# スケール正規化して比較
E_est_normalized = E_est / np.linalg.norm(E_est) * np.sign(E_est[0, 1])
E_true_normalized = E_true / np.linalg.norm(E_true) * np.sign(E_true[0, 1])

print(f"\n推定 E (8点法, σ={sigma_px}px):")
print(E_est_normalized)
print(f"\n真の E:")
print(E_true_normalized)
print(f"\n差のノルム: {np.linalg.norm(E_est_normalized - E_true_normalized):.4f}")

## 7.3 バンドル調整 (Bundle Adjustment)

**SLAM Handbook Section 7.3.3, Eq. 7.1**: カメラポーズと3D点を **再投影誤差** の最小化で同時推定。

$$E_{\text{reproj}} = \frac{1}{2} \sum_{i,j} \left\|\mathbf{z}_{ij} - \pi(\mathbf{T}_i^{-1} \mathbf{p}_j)\right\|^2_{\boldsymbol{\Sigma}_{ij}}$$

ここで $\mathbf{T}_i$ はカメラポーズ（SE(3)）、$\mathbf{p}_j$ は3Dランドマーク。

ヤコビアンの構造は **スパース** — 各観測は1つのカメラと1つの3D点のみに依存 → Ch01のスパースLS。

In [ ]:
# =============================================================
# 簡易Bundle Adjustment の実装
# =============================================================
np.random.seed(42)

# シーン設定: 4カメラ + 15個の3D点
n_cams = 4
n_points = 15

# 真のカメラポーズ (R, t)
cam_rotations = [so3_exp([0,0,0]), so3_exp([0,0.15,0]),
                 so3_exp([0,0.3,0.05]), so3_exp([0.05,0.45,0])]
cam_translations = [np.array([0,0,0]), np.array([1,0,0.1]),
                    np.array([2,0.2,0]), np.array([3,0,0.2])]

# 真の3D点（カメラの前方にランダム配置）
points_true = np.random.uniform([-2, -1.5, 3], [2, 1.5, 8], (n_points, 3))

# ノイズ付き観測を生成
sigma_obs = 1.0  # pixel
observations = []  # (cam_idx, point_idx, z_observed)

for i in range(n_cams):
    for j in range(n_points):
        p_cam = cam_rotations[i] @ points_true[j] + cam_translations[i]
        if p_cam[2] > 0.5:  # カメラの前方にある点のみ
            z = cam.project(p_cam.reshape(1, 3))[0]
            if 0 <= z[0] <= 640 and 0 <= z[1] <= 480:  # 画像内
                z_noisy = z + np.random.normal(0, sigma_obs, 2)
                observations.append((i, j, z_noisy))

print(f"カメラ数: {n_cams}, 3D点数: {n_points}, 観測数: {len(observations)}")

# BA: 状態ベクトル = [cam0_rot(3), cam0_trans(3), ..., pt0(3), pt1(3), ...]
# cam0は固定 (gauge)
def pack_state(rotations, translations, points):
    state = []
    for i in range(1, len(rotations)):  # cam0 skip
        from scipy.linalg import logm
        R = rotations[i]
        cos_phi = np.clip((np.trace(R)-1)/2, -1, 1)
        phi = np.arccos(cos_phi)
        if phi < 1e-10:
            rv = np.zeros(3)
        else:
            rv = phi/(2*np.sin(phi)) * np.array([R[2,1]-R[1,2], R[0,2]-R[2,0], R[1,0]-R[0,1]])
        state.extend(rv)
        state.extend(translations[i])
    for p in points:
        state.extend(p)
    return np.array(state)

def unpack_state(state, n_cams, n_points):
    rotations = [np.eye(3)]  # cam0 fixed
    translations = [np.zeros(3)]
    idx = 0
    for i in range(1, n_cams):
        rv = state[idx:idx+3]; idx += 3
        rotations.append(so3_exp(rv))
        translations.append(state[idx:idx+3].copy()); idx += 3
    points = []
    for j in range(n_points):
        points.append(state[idx:idx+3].copy()); idx += 3
    return rotations, translations, points

def ba_residuals(state):
    rotations, translations, points = unpack_state(state, n_cams, n_points)
    residuals = []
    for cam_i, pt_j, z_obs in observations:
        p_cam = rotations[cam_i] @ np.array(points[pt_j]) + translations[cam_i]
        z_proj = cam.project(p_cam.reshape(1, 3))[0]
        residuals.extend((z_obs - z_proj) / sigma_obs)
    return np.array(residuals)

# 初期値にノイズを加える
init_rots = [R.copy() for R in cam_rotations]
init_trans = [t.copy() for t in cam_translations]
init_points = [p.copy() for p in points_true]

for i in range(1, n_cams):
    init_rots[i] = init_rots[i] @ so3_exp(np.random.normal(0, 0.05, 3))
    init_trans[i] += np.random.normal(0, 0.2, 3)
for j in range(n_points):
    init_points[j] += np.random.normal(0, 0.3, 3)

x0 = pack_state(init_rots, init_trans, init_points)
print(f"状態ベクトル次元: {len(x0)} ({(n_cams-1)*6} camera + {n_points*3} points)")
print(f"残差次元: {len(ba_residuals(x0))} ({len(observations)}×2)")

# scipy least_squares で解く
result = least_squares(ba_residuals, x0, method='lm', verbose=0)
print(f"\n初期コスト: {0.5*np.sum(ba_residuals(x0)**2):.2f}")
print(f"最終コスト: {0.5*np.sum(result.fun**2):.2f}")
print(f"収束: {result.success}")

In [ ]:
# =============================================================
# BA結果の可視化
# =============================================================
rots_opt, trans_opt, pts_opt = unpack_state(result.x, n_cams, n_points)

fig = plt.figure(figsize=(14, 6))
ax = fig.add_subplot(121, projection='3d')
ax2 = fig.add_subplot(122, projection='3d')

for ax_curr, title, rots, trans, pts in [
    (ax, 'Before BA (初期値)', init_rots, init_trans, init_points),
    (ax2, 'After BA (最適化後)', rots_opt, trans_opt, pts_opt)]:
    
    # 真値
    ax_curr.scatter(*points_true.T, c='black', s=20, alpha=0.3, label='True 3D points')
    for i in range(n_cams):
        ax_curr.scatter(*cam_translations[i], c='black', s=50, alpha=0.3, marker='^')
    
    # 推定値
    pts_arr = np.array(pts)
    ax_curr.scatter(*pts_arr.T, c='blue', s=20, label='Estimated points')
    for i in range(n_cams):
        ax_curr.scatter(*trans[i], c='red', s=80, marker='^', label='Camera' if i==0 else '')
    
    ax_curr.set_xlabel('X'); ax_curr.set_ylabel('Y'); ax_curr.set_zlabel('Z')
    ax_curr.set_title(title, fontweight='bold')
    ax_curr.legend(fontsize=8)

plt.tight_layout(); plt.show()

# 3D点の誤差
pts_init = np.array(init_points)
pts_final = np.array(pts_opt)
err_init = np.mean(np.linalg.norm(pts_init - points_true, axis=1))
err_final = np.mean(np.linalg.norm(pts_final - points_true, axis=1))
print(f"3D点の平均誤差: 初期値 {err_init:.4f}m → BA後 {err_final:.4f}m")

## 7.4 演習問題

### 演習1: Triangulation
2つのカメラからの対応点のみから3D点を復元（triangulation）する関数を実装してください。DLT法またはmidpoint法を使います。

### 演習2: PnP問題
既知の3D点と2D観測から1つのカメラポーズを推定するPnP（Perspective-n-Point）を実装してください。

### 演習3: ヤコビアンのスパース構造
BAのヤコビアン行列のスパースパターンを可視化し、Schur Complementによるランドマーク消去を実装してみましょう。

## まとめ

| 要素 | 内容 |
|---|---|
| **Pinhole Model** | $\pi(x,y,z) = [f_u x/z + u_0, f_v y/z + v_0]$ |
| **Reprojection Error** | 観測2Dと射影2Dのピクセル差 → BA最適化の残差 |
| **Essential Matrix** | $\mathbf{E} = [\mathbf{t}]_\times \mathbf{R}$, エピポーラ制約 |
| **8点法** | 8+対応点から$\mathbf{E}$を推定 |
| **Bundle Adjustment** | カメラポーズ+3D点の再投影誤差を同時最小化 |

### 次のNotebook
→ Ch08: LiDAR SLAM (ICP, Scan Matching, Pose Graph)